# Implement Using Langchain

### Insert Documents 
 - Import the documents here
 - Chunk using the RecursiveTextSplitter
 - Use Document class to create various documents
 - attach meta data to each Document
 - use Pinecone Vector Store to store each document

In [3]:
# Importing important lib

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from datetime import date
from langchain_pinecone import PineconeVectorStore
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from pinecone import ServerlessSpec, Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_openai import ChatOpenAI
from yaml import safe_load
from langsmith import traceable

from dotenv import load_dotenv
load_dotenv()


c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [1]:
def get_chunks():
    docs_path = os.path.join(os.getcwd(), "docs")

    chunks = []
    for doc in os.listdir(docs_path):
        try:
            # Try UTF-8 first, fall back to latin-1 if it fails
            try:
                doc_content = TextLoader(os.path.join(docs_path, doc), encoding="utf-8").load()
            except (UnicodeDecodeError, LookupError):
                print(f"⚠️  UTF-8 failed for {doc}, trying latin-1...")
                doc_content = TextLoader(os.path.join(docs_path, doc), encoding="latin-1").load()

            # chunk the data using MarkdownHeaderTextSplitter
            headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
            ("####", "Header 4"),
            ("#####", "Header 5"),
            ("######", "Header 6"),]
            text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
            doc_chunks = text_splitter.split_text(doc_content[0].page_content)
            for doc_chunk in doc_chunks:
                meta_data = {
                    "source": doc,
                    "chunk_content": doc_chunk.page_content,
                    "timestamp": date.today().ctime(),
                    **doc_chunk.metadata
                }
                chunk = Document(page_content=doc_chunk.page_content, metadata=meta_data)
                chunks.append(chunk)
            print(f"✓ Loaded {doc}: {len(doc_chunks)} chunks")
        except Exception as e:
            print(f"❌ Error loading {doc}: {str(e)}")
            continue

    print(f"\n✓ Total chunks loaded: {len(chunks)}")
    return chunks

### initialise the PineconeVectorizer

In [4]:
def get_dense_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

def get_sparse_embedding_model():
    """Create BM25 sparse encoder and fit on document chunks"""
    chunks = get_chunks()
    
    if not chunks:
        raise ValueError("No chunks available for BM25 fitting. Check document loading.")
    
    bm25_encoder = BM25Encoder().default()
    # Extract text content from chunks
    texts = [chunk.page_content for chunk in chunks]
    print(f"Fitting BM25 on {len(texts)} texts...")
    bm25_encoder.fit(texts)
    return bm25_encoder

def initialize_pinecone():
    index_name = "practice-project-company-docs-hybrid"  # Changed name to force new index creation

    pc = Pinecone(api_key=os.getenv('PINECONE_KEY'))

    # Delete old index if it exists to avoid dimension mismatch
    if pc.has_index("practice-project-comapny-docs"):
        print("Deleting old index with incorrect dimension...")
        pc.delete_index("practice-project-comapny-docs")
    
    if not pc.has_index(index_name):
        print(f"Creating new index '{index_name}' with dimension 384...")
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    index = pc.Index(index_name)

    dense_embeddings = get_dense_embedding_model()
    sparse_embeddings = get_sparse_embedding_model()

    hybrid_retriever = PineconeHybridSearchRetriever(
        index=index,
        embeddings=dense_embeddings,
        sparse_encoder=sparse_embeddings,
    )

    return hybrid_retriever
    # initialize pinecone

### store documents

In [ ]:
hybrid_retriever = initialize_pinecone()

✓ Loaded company_operations.md: 62 chunks
✓ Loaded customer_service_guide.md: 42 chunks
✓ Loaded finance_procedures.md: 63 chunks
✓ Loaded hr_handbook.md: 16 chunks
✓ Loaded it_support_guide.md: 59 chunks
✓ Loaded sample_policy.md: 12 chunks
✓ Loaded technical_documentation.md: 62 chunks

✓ Total chunks loaded: 316
Fitting BM25 on 316 texts...


100%|██████████| 316/316 [00:00<00:00, 852.35it/s]


In [17]:
query = "have I asked about the leave approval process?"
# hybrid_retriever.top_k = 6
# hybrid_retriever.alpha = 0.7
# context = hybrid_retriever.invoke(query)

In [1]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.inference.ai.azure.com"
)

llm.invoke("Hello world").content

NameError: name 'ChatOpenAI' is not defined

In [8]:
def get_system_prompt():
    prompt_path = os.path.join(os.getcwd(), "prompts", "simple_system_prompt.yaml")
    with open(prompt_path, "r") as f:
        loaded_prompt = safe_load(f).get('template')
    return loaded_prompt

system_prompt = get_system_prompt()
system_prompt

'You are an AI assistant which helps answer questions regarding company policy data.\nRules:\n- If You are given a question about company policies you need to strictly adhere to the company\'s policies and guidelines.\n- If a question is asked which is not related to company policies, you should respond with "I am sorry, I can only answer questions related to company policies."\n- Always use the provided context to answer the question. Do not use any information outside of the provided context.\n- If the answer to the question is not found in the provided context, respond with "I am sorry, I do not have enough information to answer that question."\n'

In [9]:
def get_human_prompt():
    prompt_path = os.path.join(os.getcwd(), "prompts", "simple_human_prompt.yaml")
    with open(prompt_path, "r") as f:
        loaded_prompt = safe_load(f).get('template')
    return loaded_prompt
human_prompt = get_human_prompt()
human_prompt

'Answer the following question \nQUESTION: {query}\nbased on the following context retrieved from the company policy documents:\nCONTEXT: {sources}\nand the history of the previous conversations with the user:\nCONVERSATION HISTORY: {history}'

#### Create a save session history function

In [10]:
chat_map = {}
def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

#### Multi Query Expansion

In [ ]:
@traceable(name="Query Enricher", run_type="prompt")
def generate_rich_query(query: str, llm: ChatOpenAI) -> str:
    print('🚀Enriching your query')
    chat_template = ChatPromptTemplate([
        ('system', 'You are an expert in enriching the given user query. You will be given a user query you need to enrich thatto cover multiple aspects and create a well rounded question for the LLM to understand. Your Query will be directly used in RAG'),
        ('user', "{query}")
    ])
    prompt_value = chat_template.invoke({"query": query})
    enriched_query = llm.invoke(prompt_value).content
    print(f"="*55)
    print('Enriched Query::', enriched_query)
    print(f"="*55)
    return enriched_query

#### Retrieve chunks from Vector DB

In [12]:
@traceable(name="RAG Retriever", run_type="retriever")
def retrieve_chunks(query:str, hybrid_retriever) -> list:
    print('🔍Retrieving relevant chunks')
    hybrid_retriever.top_k = 6
    hybrid_retriever.alpha = 0.4
    context = hybrid_retriever.invoke(query)
    for chunk in context:
        print("-"*55)
        print(chunk)
        print("-"*55)
    return context

#### Augment chunks and query in the prompt template

In [13]:
@traceable(name="Prompt Augmenter", run_type="prompt")
def augment_into_prompt(context:list, query:str, history: list):
    print("="*55)
    print(f"QUERY: {query}")
    print("="*55)
    print('➕ adding retrieved context to the prompt')
    system_prompt = get_system_prompt()
    human_prompt = get_human_prompt()
    chat_template = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template(human_prompt)
    ])
    completed_prompt =chat_template.invoke({
        "query": query,
        "history": history,
        "sources": "\n".join([con.page_content for con in context])
    })
    for message in completed_prompt.to_messages():
        print(f"{message.type.upper()}:: {message.content}")
    return completed_prompt

#### Invoke the LLM with the augmented prompt

In [14]:
@traceable(name="LLM Invoker", run_type="llm")
def invoke_llm(prompt_value: str, llm: ChatOpenAI):
    print('🤖invoking the LLM')
    response = llm.invoke(prompt_value)
    return response

### Combine all of this together for Retrieval pipeline 

In [18]:
chain = (
    RunnablePassthrough()
    .assign(enriched_query=RunnableLambda(lambda x: generate_rich_query(x['query'], x['llm'])))
    .assign(retrieved_chunks=RunnableLambda(lambda x: retrieve_chunks(x['enriched_query'], x['hybrid_retriever'])))
    .assign(final_prompt=RunnableLambda(lambda x: augment_into_prompt(x['retrieved_chunks'], x['enriched_query'], x['history'])))
    .assign(response=RunnableLambda(lambda x: invoke_llm(x['final_prompt'], x['llm'])))
)

pipelinewith_history = RunnableWithMessageHistory(
    get_session_history=get_chat_history,
    history_messages_key='history',
    input_messages_key="query",
    runnable=chain,
    output_messages_key="response"
)


inputs = {
    "query": query,
    "llm": llm,
    "hybrid_retriever": hybrid_retriever,
    "session_id": "user_125"
}

response = pipelinewith_history.invoke(inputs, config={"configurable": { "session_id": "user_125" }})

print(response['response'].content)


🚀Enriching your query
🔍Retrieving relevant chunks
-------------------------------------------------------
page_content='Vacation (PTO): Employees accrue 1.25 days per month (15 days annually) during years 0–2; 1.67 days per month (20 days annually) during years 3–5; and 2.08 days per month (25 days annually) after year 6. Carryover up to 5 days is permitted unless local law is more generous. PTO must be requested via PeopleHub at least 10 calendar days in advance for absences of three days or more. Supervisors must ensure adequate coverage and cannot unreasonably deny requests.  
Sick Leave: Employees receive 10 paid sick days per year, accrued monthly. Use is permitted for personal illness, medical appointments, or to care for an ill family member. Documentation may be required for absences of three consecutive days. Employees must notify their manager as soon as practicable and record time in PeopleHub.  
Family & Other Leave: Parental leave provides 12 weeks paid for birthing parent

In [19]:
history = get_chat_history("user_125").messages
history

[HumanMessage(content='have I ever asked a question on technology philosophy of the company?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I am sorry, I do not have enough information to answer that question.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 905, 'total_tokens': 921, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DJcX9Q0tnRrrMgmup7HMaQQ1IVrn5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf0ef-17bf-7f71-a370-12159b8828ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 905, 'output_tokens': 16, 'total_tokens': 921, 'input_token_details': {'audio': 0, 'cache_read': 0},

: 